In [ ]:
%pip install lovelytics-cli

In [ ]:
%sh
export BROWSER=echo
love login

In [ ]:
%sh
love install python-package genie-assessment

In [ ]:
import glob
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path


dbutils.widgets.text(
    "config_workspace_path",
    "/Workspace/Users/${workspace.current_user.userName}/.bundle/template_databricks_asset_bundle/dev/files/genie_assessment/temp/config.json",
    "Config JSON path",
)

config_workspace_path = dbutils.widgets.get("config_workspace_path")
user_name = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .userName()
    .get()
)
workspace_url = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .workspaceUrl()
    .get()
)
assessment_directory = Path("/Workspace/Users") / user_name / "genie_assessment"
assessment_directory.mkdir(parents=True, exist_ok=True)
local_config_path = assessment_directory / "assessment_config.json"
workspace_config_path = Path(config_workspace_path)

if workspace_config_path.is_file():
    shutil.copyfile(workspace_config_path, local_config_path)
else:
    config_content = dbutils.fs.head(config_workspace_path)
    local_config_path.write_text(config_content, encoding="utf-8")

with local_config_path.open(encoding="utf-8") as config_file:
    config = json.load(config_file)

if not config.get("warehouse_id"):
    raise ValueError("El config no contiene warehouse_id")

os.environ["DATABRICKS_HOST"] = f"https://{workspace_url}"
os.environ["DATABRICKS_TOKEN"] = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .apiToken()
    .get()
)

script_directories = [
    Path.home() / ".local" / "bin",
    Path(sys.prefix) / "bin",
    Path("/databricks/python/bin"),
]
for directory in script_directories:
    if directory.is_dir():
        os.environ["PATH"] = f"{directory}{os.pathsep}{os.environ['PATH']}"

genie_assess_command = shutil.which("genie-assess")
if not genie_assess_command:
    candidates = [
        Path(directory) / "genie-assess"
        for directory in script_directories
    ]
    candidates.extend(Path(path) for path in glob.glob("/root/.local/bin/genie-assess"))
    genie_assess_command = next(
        (str(path) for path in candidates if path.is_file()),
        None,
    )

if not genie_assess_command:
    raise FileNotFoundError(
        "No se encontró genie-assess. Verifica que 'love install python-package "
        "genie-assessment' haya terminado correctamente. Rutas revisadas: "
        + ", ".join(str(path) for path in script_directories)
    )

result = subprocess.run(
    [genie_assess_command, "--config", str(local_config_path)],
    capture_output=True,
    text=True,
    env=os.environ.copy(),
    cwd=str(assessment_directory),
)

print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"genie-assess terminó con código {result.returncode}")